# Automatic contact inference

Run the trained contact model beside the existing reviewed pipeline. All frames are native, zero-based video frames. Read `docs/automatic_contacts.md` and the model card before accepting new-wall results.

Use the isolated environment from `requirements-contact.txt` as this notebook kernel. The full training workflow is documented separately.


In [ ]:
from pathlib import Path
import json
import numpy as np
import pandas as pd

ROOT = Path.cwd()
assert (ROOT / "beta_translator").is_dir(), "Open this notebook from the repository root."
VIDEO = ROOT / "videos/trial1.mp4"
REVIEWED_OUTPUT = ROOT / "outputs"
MODEL = ROOT / "models/contact_v1"
AUTO = ROOT / "outputs/automatic"
POSE_CACHE = AUTO / "wholebody.npz"
PREDICTIONS = AUTO / "predictions"
POSE_DEVICE = "coreml"  # Use cpu when CoreML is unavailable.
MODEL_DEVICE = "mps"  # cpu and cuda are also supported.


## 1. Extract whole-body pose

This uses the public RTMW model, including fingertips and toes. It does not train on this clip. The cache records video/model hashes and tracking version.


In [ ]:
from beta_translator.wholebody_pose import WholebodyExtractor

extractor = WholebodyExtractor(ROOT / "training_data/model_cache", POSE_DEVICE, threads=2)
extractor.extract(VIDEO, POSE_CACHE)


## 2. Associate holds and detect transitions

The reviewed hold map and camera alignment are reused. Outputs go to a separate directory. `candidate_hold` is an inference; `hold_id=unknown` means the confidence or observation checks require review.


In [ ]:
from beta_translator.automatic_contacts import run_automatic_contacts

summary = run_automatic_contacts(VIDEO, POSE_CACHE, REVIEWED_OUTPUT, MODEL, PREDICTIONS, MODEL_DEVICE)
print(f"Accepted limb-frame fraction: {summary['accepted_frame_fraction']:.1%}")
print(f"IK-eligible movement proposals: {summary['eligible_moves']}")
contacts = pd.read_csv(PREDICTIONS / "limb_contacts.csv")
events = pd.read_csv(PREDICTIONS / "contact_events.csv")
moves = pd.DataFrame(summary["moves"])
display(events, moves)


## 3. Inspect a movement

Change the frame range below. The original 117/128 reference is only a local integration check, not training data or a general accuracy estimate.


In [ ]:
start, end = 100, 140
display(contacts[contacts.frame.between(start, end)].pivot(index="frame", columns="limb", values="candidate_hold"))
display(contacts[contacts.frame.between(start, end)].pivot(index="frame", columns="limb", values="confidence"))


## 4. Prefill a review draft

Select a proposed movement by its row index. This saves a separate draft with endpoint labels and suggested timing. It never confirms support continuity or overwrites reviewed annotations. The current package keeps automatic IK disabled because its validation requirement was not met.


In [ ]:
from beta_translator.automatic_contacts import review_annotation_from_prediction

MOVE_INDEX = 0
if summary["moves"]:
    template = json.loads((REVIEWED_OUTPUT / "state/endpoint_annotations.json").read_text())
    draft = review_annotation_from_prediction(template, summary["moves"][MOVE_INDEX], summary, contacts)
    draft_path = AUTO / f"review_draft_{MOVE_INDEX}.json"
    draft_path.write_text(json.dumps(draft, indent=2) + "\n")
    print(draft_path)
    print(json.dumps(draft, indent=2))


## 5. Build an IK annotation when eligible

The adapter returns a new object. It checks input identity and requires a confident release phase plus three unchanged support contacts. Uncertain proposals stay in the review table. The template supplies measured wall scale, height, and body ratios.


In [ ]:
from beta_translator.automatic_contacts import motion_annotation_from_prediction
from beta_translator.motion_ik import contact_schedule

eligible = [m for m in summary["moves"] if m["eligible_for_ik"]]
if eligible:
    template = json.loads((REVIEWED_OUTPUT / "state/endpoint_annotations.json").read_text())
    annotation = motion_annotation_from_prediction(template, eligible[0], summary)
    frames, schedule = contact_schedule(annotation)
    print(json.dumps(annotation, indent=2))
    print(f"IK schedule: {len(frames)} frames")
else:
    print("No fully supported automatic move. Review uncertain contacts/timing before IK.")


## Evaluation

These reports hold participant-disjoint public-data results. Confidence coverage counts limb-frames, not complete moves or time saved by a human annotator. New-wall accuracy still needs independent evaluation.


In [ ]:
for filename in ("validation_report.json", "test_report.json"):
    report = json.loads((MODEL / filename).read_text())
    print(filename)
    display(pd.DataFrame({name: report[name]["aggregate"] for name in ("model", "baseline")}).T)
